# SLBusinessIndex — NeuroGrowth ML-02 / Deepnote-first
**RESEARCH ONLY.** No production DB, API credentials, customer data or real-vendor accuracy claims. This notebook reruns the existing synthetic ML-01 experiment and checks a privacy-safe feature contract. The project must have GitHub network access. A GPU is optional.

In [ ]:
import os, sys, json, pathlib, subprocess, platform
workspace = pathlib.Path.cwd()
repo = workspace / 'sl_business_index'
branch = 'research/neurogrowth-ml02-deepnote-foundation-20261125'
if not (repo / 'ml/neurogrowth/train.py').exists():
    subprocess.run(['git','clone','--depth','1','--branch',branch,'https://github.com/Victor-dev-03-cmd/sl_business_index.git',str(repo)],check=True)
print('Python:', sys.version.split()[0], 'platform:', platform.platform())
print('Repository:',repo)
print('No vendor/customer records downloaded.')

In [ ]:
try:
    import torch, numpy as np
except ImportError as exc:
    raise RuntimeError('Install PyTorch and NumPy in Deepnote project environment using ml/neurogrowth/requirements-colab.txt; do not paste credentials.') from exc
print('torch:', torch.__version__, 'numpy:', np.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
print('This is a SYNTHETIC-only run. GPU is not required.')

In [ ]:
contract = json.loads((repo/'ml/neurogrowth/contracts/ml02_research_feature_contract.json').read_text())
assert contract['research_only'] is True and contract['no_raw_pii_export'] is True
assert len(contract['features']) == 10
assert len(set(contract['features'])) == 10
assert all(f['min'] == 0 and f['max'] == 1 for f in contract['features'])
print('Feature contract:',contract['contract_version'])
print('Prohibited raw fields:', ', '.join(contract['prohibited_raw_exports']))

In [ ]:
output = repo/'ml/neurogrowth/model_artifacts/ml02_deepnote_synthetic.json'
subprocess.run([sys.executable,str(repo/'ml/neurogrowth/train.py'),'--seed','42','--size','2500','--epochs','75','--output',str(output)],cwd=repo,check=True)
metrics = json.loads(output.with_suffix('.metrics.json').read_text())
assert metrics['research_only'] and not metrics['real_business_validation']
print(json.dumps(metrics,indent=2))

In [ ]:
import hashlib
sha = hashlib.sha256(output.read_bytes()).hexdigest()
print('Synthetic checkpoint SHA256:',sha)
print('WARNING: results above evaluate generated simulator labels ONLY; they do not predict sales, leads, profits, or live business outcomes.')

## ML-03 gate: STOP before real data
Do NOT import actual vendor datasets or production database credentials into Deepnote. Real data requires documented consent, aggregation and de-identification, business/time-aware splits, privacy review, and stronger baselines (existing G2/G5 rules and gradient boosting). No model deployment or autonomous vendor dashboard actions.